# PPO

In [10]:
import gymnasium as gym
import numpy as np

import os
import random

import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical

In [21]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, checkpoint_dir = "", file_name = "policy.pth") -> None:
        super().__init__()

        self.device = "cude" if torch.cuda.is_available() else "cpu"

        self.net = nn.Sequential(
            nn.Linear(state_dim, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, action_dim)        
        ).to(self.device)

        self.checkpoint_dir = os.path.join(checkpoint_dir, file_name)

    def forward(self, x):
        return self.net(x)

    def save_checkpoint(self):
        os.makedirs(os.path.dirname(self.checkpoint_dir), exist_ok=True)
        torch.save(self.state_dict(), self.checkpoint_dir)

    def load_checkpoint(self):
        self.load_state_dict(torch.load(self.checkpoint_dir))
        self.net.to(self.device)



class ValueNetwork(nn.Module):
    def __init__(self, state_dim, checkpoint_dir = "", file_name = "value.pth") -> None:
        super().__init__()

        self.device = "cude" if torch.cuda.is_available() else "cpu"

        self.net = nn.Sequential(
            nn.Linear(state_dim, 100),
            nn.ReLU(),
            nn.Linear(100, 10),
            nn.ReLU(),
            nn.Linear(10, 1)
        ).to(self.device)

        self.checkpoint_dir = os.path.join(checkpoint_dir, file_name)


    def forward(self, x):
        return self.net(x)

    def save_checkpoint(self):
            os.makedirs(os.path.dirname(self.checkpoint_dir), exist_ok=True)
            torch.save(self.state_dict(), self.checkpoint_dir)
    
    def load_checkpoint(self):
        self.load_state_dict(torch.load(self.checkpoint_dir))
        self.net.to(self.device)

In [22]:
class RollOutMemory:
    def __init__(self) -> None:
        self.states = []
        self.actions = []
        self.rewards = []
        self.next_states = []
        self.values = []
        self.old_log_probs = []
        self.dones = []
        self.returns = []
        self.advantages = []

    def save(self, state, action, reward, next_state, value, old_log_prob, done):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.next_states.append(next_state)
        self.values.append(value)
        self.old_log_probs.append(old_log_prob)
        self.dones.append(done)

    def clear(self):
        self.states.clear()
        self.actions.clear()
        self.rewards.clear()
        self.next_states.clear()
        self.values.clear()
        self.old_log_probs.clear()
        self.dones.clear()

    def sample(self, batch_size):
        n = len(self.rewards)
        indices = torch.randperm(n)
        for start in range(0, n, batch_size):
            batch_indices = indices[start:start+batch_size]
            yield (
                [self.states[i] for i in batch_indices],
                [self.actions[i] for i in batch_indices],
                [self.rewards[i] for i in batch_indices],
                [self.next_states[i] for i in batch_indices],
                [self.values[i] for i in batch_indices],
                [self.old_log_probs[i] for i in batch_indices],
                [self.advantages[i] for i in batch_indices],
                [self.returns[i] for i in batch_indices],
                [self.dones[i] for i in batch_indices],
            )

In [ ]:
from torch import logit


class PPOAgent:
    def __init__(self,env_name,  state_dim, action_dim, policy_checkpoint_dir = "", value_checkpoint_dir = "") -> None:
        self.policy = PolicyNetwork(state_dim, action_dim, policy_checkpoint_dir)
        self.value = ValueNetwork(state_dim, value_checkpoint_dir)
        self.rollout_memory = RollOutMemory()
        self.env = gym.make(env_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.critic_loss = nn.functional.mse_loss

    def save_agent(self):
        self.policy.save_checkpoint()
        self.value.save_checkpoint()

    def load_agent(self):
        self.policy.load_checkpoint()
        self.value.load_checkpoint()

    def train(self, discount_factor = 0.99, gae_factor = 0.95, epsilon = 0.2,  rollouts = 2048, training_iters = 10, batch_size = 64, critic_factor = 0.9, entropy_factor = 0.01, ppo_iters = 50):
        for i in range(ppo_iters):
            observation, info = self.env.reset()

            # collecting rollout
            for t in range(rollouts):
                observation = torch.tensor(observation, device=self.device, requires_grad=False)

                with torch.no_grad():
                    actions_logits = self.policy(observation)
                    actions_dist = Categorical(logits=actions_logits)
                    action = actions_dist.sample()
                    action_log_prob = actions_dist.log_prob(action)

                    value = self.value(observation)

                    new_observation, reward, terminated, truncated, info = self.env.step(action.item())

                    done = terminated or truncated

                    self.rollout_memory.save(observation, action, reward, new_observation, value, action_log_prob, done)

                    if done:
                        observation, info = self.env.reset()
            # end rollout loop


            # GAE Calculation

            with torch.no_grad():
                advantage = 0.0
                for t in reversed(range(rollouts)):
                    rt = self.rollout_memory.rewards[t]
                    vt = self.rollout_memory.values[t]

                    if(t == (rollouts -1)):
                        vtnext = self.value(
                            torch.tensor(
                                self.rollout_memory.states[t],
                                device=self.device,
                                requires_grad=False
                            )
                        )
                    else:
                        vtnext = self.rollout_memory.values[t+1]

                    if self.rollout_memory.dones[t]:
                        vtnext = 0

                    delta = rt + discount_factor * vtnext - vt

                    advantage = delta + discount_factor * gae_factor * advantage

                    self.rollout_memory.advantages.append(advantage)

                    self.rollout_memory.returns.append(advantage + vt)

                self.rollout_memory.advantages.reverse()
                self.rollout_memory.returns.reverse()
            # end GAE calculation


            #PPO optimization
            for k in range(training_iters):
                for batch in self.rollout_memory.sample(batch_size):
                    states, actions, rewards, next_states, values,\
                    old_log_probs, advantages, returns, dones = batch

                    states = torch.stack(states)
                    actions = torch.stack(actions)
                    old_log_probs = torch.stack(old_log_probs)
                    advantages = torch.stack(advantages)
                    returns = torch.stack(returns)

                    logits = self.policy(states)
                    dist = Categorical(logits=logits)
                    log_probs = dist.log_prob(actions)

                    


                    ratio = torch.exp(log_probs - old_log_probs)

                    L_1 = advantages * ratio
                    L_2 = torch.clip(ratio, 1-epsilon, 1+epsilon) * advantages

                    actor_loss = (
                        torch.min(L_1, L_2)
                    ).mean()

                    
            
        

In [30]:
agent.rollout_memory.states[0]

tensor([-0.0022,  1.4035, -0.2270, -0.3288,  0.0026,  0.0514,  0.0000,  0.0000])

In [31]:
states = torch.stack(agent.rollout_memory.states)

In [34]:
states

tensor([[-0.0022,  1.4035, -0.2270,  ...,  0.0514,  0.0000,  0.0000],
        [-0.0022,  1.4035, -0.2270,  ...,  0.0514,  0.0000,  0.0000],
        [-0.0022,  1.4035, -0.2270,  ...,  0.0514,  0.0000,  0.0000],
        ...,
        [ 0.0021,  1.4035,  0.2132,  ..., -0.0483,  0.0000,  0.0000],
        [ 0.0021,  1.4035,  0.2132,  ..., -0.0483,  0.0000,  0.0000],
        [ 0.0021,  1.4035,  0.2132,  ..., -0.0483,  0.0000,  0.0000]])

In [ ]:
#  [self.states[i] for i in batch_indices],
#                 [self.actions[i] for i in batch_indices],
#                 [self.rewards[i] for i in batch_indices],
#                 [self.next_states[i] for i in batch_indices],
#                 [self.values[i] for i in batch_indices],
#                 [self.old_log_probs[i] for i in batch_indices],
#                 [self.advantages[i] for i in batch_indices],
#                 [self.returns[i] for i in batch_indices],
#                 [self.dones[i] for i in batch_indices],

In [24]:
agent = PPOAgent("LunarLander-v3", 8, 4)

In [25]:
agent.train()

C:\Users\Hasnain Ahmad\AppData\Local\Temp\ipykernel_6528\2891334466.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  observation = torch.tensor(observation, device=self.device, requires_grad=False)
C:\Users\Hasnain Ahmad\AppData\Local\Temp\ipykernel_6528\2891334466.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(


In [121]:
actions_logits = Categorical(torch.tensor([0.5, 0.5]))
actions_logits

Categorical(probs: torch.Size([2]))

In [5]:
a = []

for i in reversed(range(5)):
    a.append(i)

In [6]:
a

[4, 3, 2, 1, 0]